# Tokenizer

In this section, we are going to build a simple tokenizer that allows us to tokenize our sentences into smaller components called tokens and decode input tokens into the original string. Each token has its own unique identifier.

In the tokenizer, we have two main processes:
- **Encoding**: Chunking our sentences into smaller components called token. Each token will be represented with a unique identifier
<br>
<img height="400px" width="700px" src="images/tokenizer_encoding.png"/>
- **Decoder**: Converting token ids into the original sentence

In this notebook, we will build a tokenizer using an algorithm called **Byte Pair Encoding**, which is used in building many large language models such as ChatGPT or Gemini.


## Importing dataset

Now let's build our tokenizer vocabulary by training it with more text data

In [ ]:
urls = {
    "Education and the good life": "https://www.gutenberg.org/cache/epub/70302/pg70302.txt",
    "The School and Society": "https://www.gutenberg.org/cache/epub/53910/pg53910.txt",
    "What Is and What Might Be": "https://www.gutenberg.org/cache/epub/20555/pg20555.txt",
    "How we think": "https://www.gutenberg.org/cache/epub/37423/pg37423.txt",
    "The Reform of Education": "https://www.gutenberg.org/cache/epub/36762/pg36762.txt"
}

# Build a script that allows you to extract and construct training dataset

In [19]:
# Import book text dataset 
with open("dataset/how_we_think.txt", 'r', encoding='utf-8-sig') as f:
    lines = f.readlines()
    # Merge those lines together
    text = "".join(lines)
    

### Byte Pair Encoding

#### Materials
- [Byte Pair Encoding Hugging Face](https://www.youtube.com/watch?v=HEikzVL-lZU)

#### Core ideas

**Rule 1**: Do not split frequently used words into smaller subwords

**Rule 2**: Split the rare words into smaller, meaningful subwords

- Eg: "play" should not be split. "playing" should be split into "play" and "ing"

#### Benefits
1. "plays" and "playing" comes from the same root "play"
2. Some words have different root words but share the suffix part such as "classification" and "location" which both share "cation"

**BFE algorithm**: Most common pair of consecutive bytes of data is replaced with a byte that does not occur in the data

In [23]:
# Build a Tokenizer Class
class Tokenizer:
    def __init__(self):
        self.token_to_id = {'<start>' : 1, '<end_of_text>' : 2, ' ' : 3, '<unk>' : 4}
        self.id_to_token = {1 : '<start>', 2 : '<end_of_text>', 3 : ' ', 4: "<unk>"}
        self.vocab = set(['<start>', '<end_of_text>', '<unk>', ' '])
        self.bp_merges = {}

    def train(self, text):
        assert len(text) > 0, "You must input a non-empty text"
        text_characters = []
        for char in text:
            if char not in self.vocab:
                self.vocab.add(char)
                id = len(self.vocab)
                self.token_to_id[char] = id
                self.id_to_token[id] = char
            text_characters.append(char)

        while True:
            frequency = {}
            # Contruct frequency from the text_characters
            for i in range(1, len(text_characters)):
                pair = (text_characters[i-1], text_characters[i])
                if pair not in frequency:
                    frequency[pair] = 0
                frequency[pair] += 1
            if not frequency: break

            most_commmon_pair, occurence = max(frequency.items(), key = lambda item: item[1])
            if occurence > 1:
                new_token = ''.join(most_commmon_pair)
                self.vocab.add(new_token)
                id = len(self.vocab)
                self.token_to_id[new_token] = id
                self.id_to_token[id] = new_token
                self.bp_merges[most_commmon_pair] = id # id here is the rank for our pair
                # Merge those tokens inside the text_characters
                new_text_characters = []

                index = 0
                while (index < len(text_characters)):
                    if (text_characters[index] == most_commmon_pair[0] and index < len(text_characters) - 1 and text_characters[index+1] == most_commmon_pair[1]):
                        new_text_characters.append(new_token)
                        index += 2 # Skip the next character
                    else:
                        new_text_characters.append(text_characters[index])
                        index += 1
                text_characters = new_text_characters
            else:
                break

    def encode(self, text: str, add_special_tokens = True):
        assert self.bp_merges, "You must train your tokenizer first!"
        tokens = list(text)
        # Merge based on the rank that a pair was constructed
        while True:
            best_rank = float("inf")
            best_pair = None
            candidate_index = -1
            for i in range(len(tokens) - 1):
                pair = (tokens[i], tokens[i+1])
                rank = self.bp_merges.get(pair, None)
                if rank is not None and rank < best_rank:
                    best_pair = pair
                    best_rank = rank
                    candidate_index = i
            if best_pair is None: break
            # Merge pair with lowest rank
            tokens[candidate_index] = ''.join(best_pair)
            del tokens[candidate_index + 1]

        ids = [self.token_to_id.get(token, self.token_to_id['<unk>']) for token in tokens]
        if add_special_tokens:
            ids = [self.token_to_id["<start>"]] + ids + [self.token_to_id['<end_of_text>']]

        return ids

    def decode(self, inputs):
        string =  "".join([self.id_to_token.get(id, self.id_to_token[4]) for id in inputs])
        return string

In [ ]:
a = [i for i in range(400000)]


In [24]:
tokenizer = Tokenizer()
# Train tokenizer
tokenizer.train(text)

In [ ]:
import pickle
import os
# Save tokenizer data
def save_tokenizer(tokenizer: Tokenizer):
    
    tokenizer_path = "tokenizer"
    if not os.path.isdir(tokenizer_path):
        os.mkdir(tokenizer_path)

    with open(f"{tokenizer_path}/token_to_id.pkl", 'wb') as f:
        pickle.dump(f, tokenizer.token_to_id)

    with open(f"{tokenizer_path}/id_to_token.pkl", "wb") as f:
        pickle.dump(f, tokenizer.id_to_token)

    with open(f"{tokenizer_path}/bp_merges.pkl", "wb") as f:
        pickle.dump(f, tokenizer.bp_merges)

def load_tokenizer(tokenizer_path = 'tokenizer'):
    with open(f"{tokenizer_path}/token_to_id.pkl", "wb") as f:
        tokenizer.token_to_id = pickle.load(f)

    with open(f"{tokenizer_path}/id_to_token.pkl", "wb") as f:
        tokenizer.id_to_token = pickle.load(f)

    with open(f"{tokenizer_path}/bp_merges.pkl", "wb") as f:
        tokenizer.bp_merges = pickle.load(f)

In [25]:
sentence = "The lesson here is under the"
ids = tokenizer.encode(sentence)
[tokenizer.id_to_token[id] for id in ids]

['<start>',
 'The ',
 'lesson ',
 'h',
 'ere is ',
 'under',
 ' the',
 '<end_of_text>']

In [27]:
# ids = [1, 59, 109, 2590, 1275, 66, 257, 25, 2]
tokenizer.decode(ids)

'<start>The lesson here is under the<end_of_text>'


### Positional encoding

- Implement positional encoding: [Link](https://kazemnejad.com/blog/transformer_architecture_positional_encoding/)

In [16]:
# Implement Positional Encoding Layer that takes in input and return the output with added position informatio
import torch
import torch.nn as nn

class PositionalEncoding(nn.Module):
    def __init__(self, d_model = 512, context_size = 1024):
        super().__init__()

        i = (torch.arange(d_model) // 2).view(1, -1)
        positions = torch.arange(context_size).view(-1, 1)

        self.encoding = positions / (10000 ** (2 * i / d_model))
        self.encoding[:, 0::2] = torch.sin(self.encoding[:, 0::2])
        self.encoding[:, 1::2] = torch.cos(self.encoding[:, 1::2])
        self.encoding = self.encoding.unsqueeze(0)
    def forward(self, X):
        # X with shape: B x N x d_model
        N = X.shape[1]
        return X + self.encoding[:, :N, :]


In [18]:
positional_encoding = PositionalEncoding()

inputs = torch.randn(10, 100, 512)

positional_encoding(inputs).shape

torch.Size([10, 100, 512])

### Create input-target pair for training

In [48]:
from torch.utils.data import Dataset, DataLoader, random_split

class EducationDataset(Dataset):
    def __init__(self, text: str, tokenizer: Tokenizer, context_size = 1024):
        self.context_size = context_size
        self.tokenizer = tokenizer
        # Naive approach first
        self.X = []
        self.y = []
        
        tokens = self.tokenizer.encode(text)
        print("token len: ", len(tokens)) 
        if len(tokens) <= context_size:
            self.X.append(tokens[:len(tokens)])
            self.y.append(tokens[1:])
        else:
            for i in range(len(tokens) - context_size + 1):
                self.X.append(tokens[i:context_size])
                print(self.X)
                self.y.append(tokens[i+1:context_size+1])
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, index):
        x, y = self.X[index], self.y[index]
        return torch.tensor(x), torch.tensor(y)

In [49]:
len(tokenizer.vocab)

14164

In [ ]:
dataset = EducationDataset(text[:20000], tokenizer)

# Split into training and testing dataset
train_ratio = 0.8
test_ratio = 1 - train_ratio
train_size = int(len(dataset) * train_ratio)
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

batch_size = 4
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Number of train batches: ", len(train_loader))
print("Number of test batches: ", len(test_loader))




### MultiHead - Self-attention mechanism

- Implement self-attention with Q, K, V approach

→ Revise on how to implement Masked multi-head attention which prevents the model from attending to subsequent time step from the current time step


In [ ]:
# Set contant 
D_MODEL = 512
NUM_HEADS = 8
D_K = D_V = D_MODEL / NUM_HEADS
D_FF = 2048


In [6]:
torch.randn(10, 8)

tensor([[ 0.5120, -0.1135,  1.1033,  0.9914, -0.7434,  0.6902, -1.8027, -0.9269],
        [ 0.1387,  0.8977,  0.4437,  0.0829,  0.8119,  0.3929, -0.2193, -1.4382],
        [-0.8758,  0.0296, -0.6644, -0.1778, -0.6625,  0.7844,  0.7021, -0.1648],
        [ 1.9041, -0.7983, -0.7790,  1.0985,  1.2654,  0.4380, -0.4766,  0.5049],
        [-1.2543, -0.2769, -1.7822,  0.4570, -0.2483, -0.7206,  0.4801, -0.8883],
        [-0.1446, -1.0059, -0.9696,  0.5869,  0.5299,  0.3289, -0.6212, -1.7981],
        [-1.4393,  1.3607,  2.2384, -1.4051,  0.4386, -0.1037,  0.3190,  0.2379],
        [ 0.8793,  0.1410,  0.0417, -0.8243, -0.0457,  0.5050, -0.0375, -0.1255],
        [-0.3235,  0.4691,  0.4512, -0.1094,  0.0170, -0.5759, -1.5575,  1.9583],
        [ 0.5311, -0.1685, -0.8801, -1.6621, -0.3780, -0.4930, -0.9380, -1.0022]])

In [ ]:
class Self_Attention(nn.Module):
    def __init__(self, d_model, d_k, d_v):
        super().__init__()
        self.d_k = d_k
        self.W_Q = torch.randn(d_model, d_k)
        self.W_K = torch.randn(d_model, d_k)
        self.W_V = torch.randn(d_model, d_v)
        
    def forward(self, X):
        Q = X@self.W_Q
        K = X@self.W_K
        V = X@self.W_V
        return torch.softmax(Q@K.T / self.d_k) * V


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, d_k, d_v, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.attentions = [Self_Attention(d_model, d_k, d_v) for i in range(self.num_heads)]
        self.W_0 = torch.randn(num_heads * d_v, d_model)
    def forward(self, X):
        output = torch.vstack([attention(X) for attention in self.attentions])
        return output @ self.W_0


class MLP(nn.Module):
    def __init__(self, input_dim, inner_dim):
        super().__init__()
        self.inner_dim = inner_dim
        self.W1 = torch.randn(input_dim, inner_dim)
        self.b1 = torch.randn(inner_dim)
        self.W2 = torch.randn(inner_dim, input_dim)
        self.b2 = torch.randn(input_dim)
        
    def forward(self, X):
        return (nn.ReLU()(X@self.W1 + self.b1))@self.W2 + self.b2
        


class Decoder(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def forward(self, X):

        pass

In [4]:
class MiniGPT(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        
    def forward(self, X):
        pass


# 3. Training model

**Train your model**

- Pretrained model for a few peochs
- Log training/validation loss and compute **perplexity**.
- Save checkpoints and final model.

**Generate text samples:**

- Use your trained model to generate coherent text related to your chosen domain.
- Show 3–5 examples with different prompts.
- Optionally experiment with **temperature, top-k, top-p sampling**

In [ ]:
EPOCHS = 5
LR = 1e-3
device = 'cpu'
model = MiniGPT()
optimizer = torch.optim.Adam(model.parameters(), lr = LR)
criterion = nn.CrossEntropyLoss()

def train_step(data_loader, model, optimizer):
    model.train()
    total_loss = 0
    for inputs, labels in data_loader:
        inputs = inputs.to(device = device)
        labels = labels.to(device = device)

        raw_logits = model(inputs)
        loss = criterion(raw_logits, labels)

        total_loss += loss.item()
        # Add perplexity metric or BLEU here

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return total_loss / len(data_loader)


def eval_step(data_loader, model):
    model.eval()
    with torch.inference_mode():
        total_loss = 0
        for inputs, labels in data_loader:
            inputs = inputs.to(device=device)
            labels = labels.to(device=device)

            raw_logits = model(inputs)
            loss = criterion(raw_logits, labels)
            total_loss += loss.item()
            
            # Add perplexity metric or BLEU here
        return total_loss / len(data_loader)

In [ ]:
for epoch in range(EPOCHS):
    train_loss = train_step(train_loader, model, optimizer)
    val_loss = eval_step(val_loader, model)
    # Logging model performance 
    print(f"Epoch: [{epoch}|{EPOCHS}]: Train loss = {train_loss} - Validation loss = {val_loss}")
    


## **4. Evaluation:**

- Report quantitative results (loss, perplexity).
- Qualitative evaluation: human judgment of coherence and relevance.